# 03. Lecke - Ügynöki Tervezési Minták

Ebben a leckében három alapvető tervezési mintát vizsgálunk meg az eredményes mesterséges intelligencia ügynökök létrehozásához:

1. **Világos Ügynöki Utasítások** — Precíz, szerepet meghatározó promptok készítése, amelyek irányítják az ügynök viselkedését  
2. **Strukturált Kimenet Pydantic Modellekkel** — Biztosítva, hogy az ügynökök kiszámítható, validált adatokat szolgáltassanak  
3. **Egyszeres Felelősségű Ügynökök** — Olyan fókuszált ügynökök tervezése, amelyek mindegyike egy feladatot végez jól

Minden mintát egy **utazási célpont ajánló** forgatókönyvre alkalmazunk, fokozatosan építve egy rendszert, amely képes javasolni célpontokat, ellenőrizni az elérhetőséget és kezelni a logisztikát.


## Beállítás


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Minta 1: Világos Ügynöki Utasítások

A leghatásosabb minta egyben a legegyszerűbb is: világos, részletes utasítások írása az ügynök számára.

A jó utasítások meghatározzák:
- **Ki** az ügynök (személyiség és stílus)
- **Mit** kell tennie (lépésről lépésre a feladatok)
- **Hogyan** viselkedjen (korlátok és stílus)

Az alábbiakban egy utazási concierge ügynököt hozunk létre explicit utasításokkal, amelyek alakítják minden általa adott választ.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Minta 2: Strukturált kimenet Pydantic modellekkel

A szabad formátumú szöveg hasznos a párbeszédhez, de a további rendszereknek strukturált adatokra van szükségük.
A **Pydantic modellek** és egy **eszközfüggvény** párosításával képesek vagyunk:

- Meghatározni a ügynök kimenetének pontos sémáját
- Automatikusan érvényesíteni a válaszokat
- Megbízhatóan integrálni az ügynök eredményeit az alkalmazáslogikába

Bevezetünk egy eszközt is, amely visszaadja a célállomás adatait, így az ügynök valós adatokra alapozhatja a javaslatait.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Mintázat 3: Egyfelelősségű Ügynökök

Az összetett feladatok előnyös, ha a munkát több, egyetlen felelősségre szakosodott ügynökre osztjuk:

- Egy **Úticél Szakértő**, aki ismeri a helyszíneket és az elérhetőséget
- Egy **Logisztikai Tervező**, aki kezeli a járatokat, a szállodákat és az útvonalakat

Ez tükrözi a szoftvermérnöki elvet, a *felelősségek szétválasztását* — így minden ügynök könnyebben tesztelhető, karbantartható és fejleszthető önállóan.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Összefoglaló

Ebben a leckében három ügynöki tervezési mintát alkalmaztunk egy utazási ajánló forgatókönyvben:

| Minta | Fő gondolat | Előny |
|---|---|---|
| **Világos utasítások** | Határozzuk meg előre a személyiséget, felelősségeket és korlátokat | Következetes, márkára szabott ügynöki viselkedés |
| **Strukturált kimenet** | Használjunk Pydantic modelleket válaszformátumként | Érvényesített, géppel olvasható eredmények |
| **Egységes felelősség** | Adjunk minden ügynöknek egy jól fókuszált feladatot | Könnyebb tesztelni, karbantartani és összerakni |

Ezek a minták természetesen kombinálhatók — az egységes felelősségű ügynökön belül világos utasításokat és strukturált kimenetet alkalmazva robusztus, éles környezetre kész rendszereket építhetünk.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
